# Day 4: PHI Governance & De-Identification Engine

This notebook demonstrates applying Format-Preserving Encryption (FPE) for SSN/MRN using `EHDIP_HASH_SALT`, deterministic date-shifting (+/- 30 days), and free-text Safe Harbor 18 redaction on synthetic data.

In [ ]:
!pip install pyspark
import os
import hashlib
from datetime import datetime, timedelta

os.environ['EHDIP_HASH_SALT'] = 'test_salt_123'
os.environ['SPARK_VERSION'] = '3.5'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, regexp_replace, lit
from pyspark.sql.types import StringType, DateType

spark = SparkSession.builder.appName("Day04_Deid_Demo").getOrCreate()
print("Spark session created")

In [ ]:
# UDFs for De-identification
salt = os.environ.get("EHDIP_HASH_SALT", "default_salt")

def fpe_hash_udf(val):
    if not val: return None
    h = hashlib.sha256(f"{val}{salt}".encode('utf-8')).hexdigest()
    return f"DEID-{h[:10]}"

def date_shift_udf(val, pid):
    if not val or not pid: return None
    stable_hash = int(hashlib.sha256(pid.encode('utf-8')).hexdigest(), 16)
    shift_days = (stable_hash % 61) - 30
    return val + timedelta(days=shift_days)

fpe_hash = udf(fpe_hash_udf, StringType())
date_shift = udf(date_shift_udf, DateType())

# Create Synthetic DataFrame
data = [
    ("P001", "123-45-6789", "MRN991", datetime(2023, 1, 15), "Patient admitted. SSN: 123-45-6789. History of HTN."),
    ("P002", "987-65-4321", "MRN992", datetime(2023, 5, 20), "Follow up. Patient feels better. No issues.")
]
columns = ["patient_id", "patient_ssn", "patient_mrn", "admission_date", "clinical_notes"]

df = spark.createDataFrame(data, columns)
print("--- RAW DATA ---")
df.show(truncate=False)

# Apply De-identification
deid_df = df \
    .withColumn("patient_ssn_deid", fpe_hash(col("patient_ssn"))) \
    .withColumn("patient_mrn_deid", fpe_hash(col("patient_mrn"))) \
    .withColumn("admission_date_shifted", date_shift(col("admission_date"), col("patient_id"))) \
    .withColumn("clinical_notes_redacted", regexp_replace(col("clinical_notes"), r'\d{3}-\d{2}-\d{4}', '[REDACTED_SSN]')) \
    .drop("patient_ssn", "patient_mrn", "admission_date") # Drop original PHI

print("--- DE-IDENTIFIED DATA ---")
deid_df.show(truncate=False)
